In [1]:

!pip install langchain langchain-community langchain-huggingface faiss-cpu chromadb pypdf groq -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7

In [2]:
!pip install langchain-groq -q
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS, Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.7 MB/s eta 0:00:00


In [3]:
class MultiPDFLoader:
    def __init__(self, file_paths):
        self.file_paths = file_paths

    def load(self):
        all_docs = []

        for path in self.file_paths:
            loader = PyPDFLoader(path)
            docs = loader.load()

            for doc in docs:
                doc.metadata["source"] = path

            all_docs.extend(docs)

        print(f"Total documents loaded: {len(all_docs)}")
        return all_docs

In [4]:
class DocumentChunker:
    def __init__(self):
        self.splitter_small = RecursiveCharacterTextSplitter(
            chunk_size=500, chunk_overlap=100
        )

        self.splitter_large = RecursiveCharacterTextSplitter(
            chunk_size=1500, chunk_overlap=300
        )

    def split(self, docs):
        chunks_small = self.splitter_small.split_documents(docs)
        chunks_large = self.splitter_large.split_documents(docs)

        print(f"Small chunks: {len(chunks_small)}")
        print(f"Large chunks: {len(chunks_large)}")

        return chunks_small, chunks_large

In [6]:
class EmbeddingManager:
    def __init__(self):
        self.embed_large = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-mpnet-base-v2"
        )

        self.embed_small = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )

In [7]:
class VectorStoreManager:
    def __init__(self, embed_large, embed_small):
        self.embed_large = embed_large
        self.embed_small = embed_small

    def create_stores(self, chunks_large, chunks_small):
        faiss_db = FAISS.from_documents(chunks_large, self.embed_large)

        chroma_db = Chroma.from_documents(
            documents=chunks_small,
            embedding=self.embed_small,
            persist_directory="./chroma_db"
        )

        return faiss_db, chroma_db

In [8]:
class RetrieverManager:
    def __init__(self, faiss_db, chroma_db):
        self.faiss_retriever = faiss_db.as_retriever(search_kwargs={"k": 3})
        self.chroma_retriever = chroma_db.as_retriever(search_kwargs={"k": 3})

    def retrieve(self, query):
        docs1 = self.faiss_retriever.invoke(query)
        docs2 = self.chroma_retriever.invoke(query)
        return docs1 + docs2

In [9]:
class LLMManager:
    def __init__(self, api_key):
        os.environ["GROQ_API_KEY"] = api_key

        self.llm = ChatGroq(
            model="llama-3.1-8b-instant",
            temperature=0.5
        )

    def generate(self, prompt):
        return self.llm.invoke(prompt).content

In [10]:
class PromptManager:
    def __init__(self):
        self.prompt = ChatPromptTemplate.from_template("""
You are a technical assistant for Visual Studio.

Answer ONLY from the given context.
If answer not found, say "Not available in document".

Context:
{context}

Question:
{question}

Answer:
""")

    def format(self, context, question):
        return self.prompt.invoke({
            "context": context,
            "question": question
        })

In [11]:
class MultiPDFRAG:
    def __init__(self, file_paths, api_key):

        # Load
        loader = MultiPDFLoader(file_paths)
        docs = loader.load()

        # Chunk
        chunker = DocumentChunker()
        chunks_small, chunks_large = chunker.split(docs)

        # Embeddings
        embed = EmbeddingManager()

        # Vector DB
        vector_store = VectorStoreManager(
            embed.embed_large,
            embed.embed_small
        )

        faiss_db, chroma_db = vector_store.create_stores(
            chunks_large, chunks_small
        )

        # Retriever
        self.retriever = RetrieverManager(faiss_db, chroma_db)

        # LLM
        self.llm = LLMManager(api_key)

        # Prompt
        self.prompt = PromptManager()

    def format_docs(self, docs):
        formatted = []
        for doc in docs:
            source = doc.metadata.get("source", "unknown")
            formatted.append(f"[Source: {source}]\n{doc.page_content}")

        return "\n\n".join(formatted)

    def ask(self, query):
        docs = self.retriever.retrieve(query)
        context = self.format_docs(docs)
        final_prompt = self.prompt.format(context, query)
        return self.llm.generate(final_prompt)

In [ ]:
file_paths = [
    "/content/technical_document1.pdf",
    "/content/technical_document2.pdf",
    "/content/technical_document3.pdf",
    "/content/technical_document4.pdf",
    "/content/technical_document5.pdf",
    "/content/technical_document6.pdf",
    "/content/technical_document7.pdf",
    "/content/technical_document8.pdf",
    "/content/technical_document9.pdf",
    "/content/technical_document10.pdf",
]

api_key = ""

rag = MultiPDFRAG(file_paths, api_key)

for i in range(20):
    q = input(f"\nQuestion {i+1}: ")
    ans = rag.ask(q)
    print("\nAnswer:\n", ans)

Total documents loaded: 1843
Small chunks: 10094
Large chunks: 3768


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Question 1: what is visual studio?

Answer:
 Visual Studio is a powerful integrated development environment (IDE) for Windows where you can develop, build, debug, test, and deploy your apps, all in one place.

Question 2: Why use Visual Studio?

Answer:
 You can discover Visual Studio, develop your code, and develop with AI.

Question 3: what is request_token ?

Answer:
 This POST requests a new token from the Pexip Conferencing Node.

Question 4: Explain PIN protected conferences

Answer:
 A PIN-protected conference is a type of conference in Pexip Infinity where participants must enter a secure access code, known as a PIN, before they can join the conference. This PIN is set by the administrator and is used to authenticate participants and determine their role in the conference, either as a Host or a Guest.

In a PIN-protected conference, the following rules apply:

- All participants must enter a valid PIN before they can join the conference.
- The PIN determines whether a particip